In [ ]:
#SFTP 

# # Use
# cnopts = pysftp.CnOpts()
# cnopts.hostkeys = None


# #Get Data from PS Data Export Manager SFTP folder, and move to BigQuery
# with pysftp.Connection(
#     host="sftp.illuminateed.com",
#     username="greendottn",
#     password="*********",
#     cnopts=cnopts
# ) as sftp:
#     # Download a remote file to the local machine
#     remote_file = "/greendottn/custom_report_standards_2024"
#     local_file = "local_file.csv"
#     sftp.get(remote_file, local_file)



# Create Bucket, Upload CSV to Bucket, Download CSV From Bucket

In [ ]:
#blob name is the name of the file once uploaded
#file_path is local file path
#bucket name is GC bucket unique bucket name

%load_ext autoreload
%autoreload 2

from modules.buckets import create_bucket, upload_to_bucket, download_from_bucket
import os

# Set the GOOGLE_APPLICATION_CREDENTIALS environment variable in order to interact
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 'greendotdataflow-848329b50f47.json'


#Create the bucket, and upload to that bucket. If already created, bypass
create_bucket('psholdingbucket6')

#Upload file to bucket, demonstrates if overwritten or newfile. 
#End File Name,  Local File Path, Bucket Name
upload_status = upload_to_bucket('students.csv' , 'Student_Record.csv', 'psholdingbucket5' )

if upload_status == 'Error': 
    print('File not uploaded, download did not occur')
    pass
else:
    # source_blob_name, destination_file_path, bucket_name
    #Input logic if file is overwritten
    download_from_bucket('students.csv', 'students_download.csv', 'psholdingbucket5')

In [ ]:
#Connect to said SFTP and get file

#We upload to Buckets through Python (creates a must use of Python if this route is taken), 

#create scheduler recognizing the CSV upload, truncate the table. 

#create next scheduler that uploads clean table from the bucket

#create next scheduler that runs SQL query to create said table. 

#create next scheduler that exports said table back to Google Cloud Storage Buckets

#Download the newly created file to Clevers SFTP




#Goal is to simplify as much as possible
#pandas_gbq removes the need for most of this other than the SQL table creation for Clever Still must be hosted


#The benefits of going through the bucket first, is that the csvs can be maintained in storage. 

# Upload From Local Storage and Specify Schema

In [25]:
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud.exceptions import Conflict

# Set the path to your service account key file
credentials_path = 'greendotdataflow-848329b50f47.json'

# Create a BigQuery client with the service account credentials
credentials = service_account.Credentials.from_service_account_file(credentials_path)
client = bigquery.Client(credentials=credentials)



table_id = 'greendotdataflow.Students.Student_Records_5'

#The schema portion is unecessary if the proper dtypes are assigned in the pandas frame
schema = [
    bigquery.SchemaField("student_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("grades", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("class", "STRING", mode="REQUIRED"),
]

table = bigquery.Table(table_id, schema=schema)

try:

    table = client.create_table(table)  # Make an API request.
    print(
        "Created table {}.{}.{}".format(table.project, table.dataset_id, table.table_id)
    )

except Conflict as e:
    print(f'Table creation failed, table already exists {table_id}')



Table creation failed, table already exists greendotdataflow.Students.Student_Records_5


# Upload From Local Storage to Created Table, Schema is Automaticaly Identified in Created Table if Not Append

In [26]:
import pandas as pd
import pandas_gbq


#DB,tablename,
table_id = 'greendotdataflow.Students.Student_Records_6'

# Read CSV file into a Pandas DataFrame
df = pd.read_csv('Student_Records.csv')
#Recognizes Schema of Pandas Frame, and Applies D-Types that way
df['student_id'] = df['student_id'].astype(str)
df['grades'] = df['grades'].astype(str)

# Upload DataFrame to BigQuery
pandas_gbq.to_gbq(df, table_id , project_id='greendotdataflow', if_exists='append', location='us-south1')



100%|██████████| 1/1 [00:00<?, ?it/s]


# Use SQL Query to Pull Down Table

In [27]:
SQL = '''
SELECT * FROM greendotdataflow.Students.Student_Records_6
'''

pandas_gbq.read_gbq(SQL, project_id='greendotdataflow', location='us-south1')


Downloading: 100%|██████████|


,student_id,grades,class
0,1,90,Math
1,5,88,Math
2,9,87,Math
3,4,78,English
4,7,82,English
5,2,85,History
6,8,91,History
7,10,89,History
8,3,92,Science
9,6,95,Science


# Read CSV File From Bucket

In [ ]:
# import bigframes.pandas as bpd

# filepath_or_buffer = "gs://cloud-samples-data/bigquery/us-states/us-states.csv"
# df_from_gcs = bpd.read_csv(filepath_or_buffer)
# # Display the first few rows of the DataFrame:
# df_from_gcs.head()